# Decision models as rerankers, on a free Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hev/reranker/blob/main/notebooks/colab_free_tier.ipynb)

A decision model takes typed questions about a piece of text and returns calibrated probabilities instead of
generated text. Asked *"is this document relevant to the query?"*, it is a reranker. This notebook runs one
**on the free Colab T4, with no API keys**: [`hev-rerank`](https://github.com/hev/reranker) with
`model="local:..."` downloads open weights from Hugging Face and runs them with `transformers` on the GPU.

It reranks BM25 results for SciFact queries in all three request shapes:

- **pair**: one request per (query, document), one yes/no probability each;
- **batch**: the query and all candidates in one request, one yes/no question per document;
- **choice**: one request, one question *"which document is best?"*, probabilities that sum to one.

**Model:** [`perplexity-ai/pplx-decider-flash-v1.3`](https://huggingface.co/perplexity-ai/pplx-decider-flash-v1.3),
a 4B decider distilled from Perplexity's 27B one (Apache-2.0). Its 8.5 GB of weights fit the T4's 16 GB in
float16 (the T4 has no bfloat16; `hev-rerank` loads float16 there automatically).

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*. The whole notebook
takes about 15 minutes, a third of it installing, downloading and loading the model.

The hosted version (Jev, Perplexity, Cloudflare Clef, OpenAI; needs keys) is
[hosted_walkthrough.ipynb](https://colab.research.google.com/github/hev/reranker/blob/main/notebooks/hosted_walkthrough.ipynb).

In [ ]:
# Parameters. HEV_RERANK_REF is the git tag or commit of hev-rerank to install.
HEV_RERANK_REF = "v0.3.0"
MODEL = "perplexity-ai/pplx-decider-flash-v1.3"
N_QUERIES = 15      # SciFact test queries per shape: sized so the notebook finishes in ~15 minutes on a T4
K = 20              # BM25 candidates per query to rerank
BATCH_DOCS = 5      # documents per batch request on the T4 (see the note under the results)

In [ ]:
%pip install -q "hev-rerank[local] @ git+https://github.com/hev/reranker@{HEV_RERANK_REF}" rank_bm25
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # fewer out-of-memory errors from fragmentation

In [ ]:
import torch, transformers, hev_rerank
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run all again."
gpu = torch.cuda.get_device_name(0)
print(f"hev-rerank {hev_rerank.__version__}, torch {torch.__version__}, transformers {transformers.__version__}")
print(f"GPU: {gpu}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB, "
      f"bf16: {torch.cuda.is_bf16_supported(including_emulation=False)}")

## Load the model

The first call downloads the weights into the Hugging Face cache (8.5 GB) and loads them on the GPU. After that
every `rerank(..., model="local:...")` call reuses the loaded model. No token is needed: the weights are public.

In [ ]:
import time
from hev_rerank import local, rerank

t0 = time.perf_counter()
model = local.connect(MODEL)
load_seconds = time.perf_counter() - t0
print(model)
print(f"download + load: {load_seconds:.0f} s, GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## Warm-up: six passages, three shapes

In [ ]:
query = "does vitamin D supplementation improve bone density in older adults"
docs = [
 "Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone density in older adults.",
 "Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many older adults fall short.",
 "Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.",
 "Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineralization.",
 "Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults over 65 versus placebo.",
 "Astronauts lose 1 to 2 percent of bone density per month in microgravity.",
]

for style in ["pair", "batch", "choice"]:
    hits = rerank(query, docs, model=f"local:{MODEL}", style=style)
    print(f"\n{style}:")
    for h in hits:
        print(f"  {h.score:6.3f}  [{h.index}] {docs[h.index][:90]}")

Pair and batch scores are independent probabilities ("how likely is *this* document relevant?"), so several
documents can score near 1. Choice spreads one unit of probability across the list: it ranks, but a choice score
says nothing about a document on its own.

## SciFact: BM25, then rerank

In [ ]:
import csv, io, json, math, random, re, urllib.request, zipfile
from rank_bm25 import BM25Okapi

# SciFact from BEIR (5,183 abstracts, 300 test queries with human relevance labels), 2.8 MB.
raw = urllib.request.urlopen("https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip").read()
z = zipfile.ZipFile(io.BytesIO(raw))
corpus = {d["_id"]: {"title": d["title"], "text": d["text"]} for d in map(json.loads, z.open("scifact/corpus.jsonl"))}
queries = {q["_id"]: q["text"] for q in map(json.loads, z.open("scifact/queries.jsonl"))}
qrels = {}
for row in csv.DictReader(io.TextIOWrapper(z.open("scifact/qrels/test.tsv")), delimiter="\t"):
    qrels.setdefault(row["query-id"], {})[row["corpus-id"]] = int(row["score"])

tok = lambda s: re.findall(r"\w+", s.lower())
ids = list(corpus)
bm25 = BM25Okapi([tok(corpus[i]["title"] + " " + corpus[i]["text"]) for i in ids])

def bm25_top(q, k=K):
    scores = bm25.get_scores(tok(q))
    return [ids[i] for i in sorted(range(len(ids)), key=lambda i: -scores[i])[:k]]

def ndcg10(ranked, rel):
    dcg = sum(rel.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked[:10]))
    ideal = sum(r / math.log2(i + 2) for i, r in enumerate(sorted(rel.values(), reverse=True)[:10]))
    return dcg / ideal if ideal else 0.0

qids = sorted(qrels)[:N_QUERIES]          # the first N test queries, sorted by id: the same subset every run
shortlists = {q: bm25_top(queries[q]) for q in qids}
bm25_ndcg = sum(ndcg10(shortlists[q], qrels[q]) for q in qids) / len(qids)
recall = sum(len(set(shortlists[q]) & set(qrels[q])) / len(qrels[q]) for q in qids) / len(qids)
print(f"{len(qids)} SciFact test queries, BM25 top-{K}: nDCG@10 {bm25_ndcg:.3f}, recall@{K} {recall:.3f} "
      "(a reranker can only reorder what BM25 found)")

In [ ]:
import statistics

def pct(xs, p):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(round(p / 100 * (len(xs) - 1))))]

results = {}
for style in ["pair", "batch", "choice"]:
    latencies, ndcgs, t0 = [], [], time.perf_counter()
    for q in qids:
        cands = [corpus[d] for d in shortlists[q]]
        torch.cuda.synchronize(); t = time.perf_counter()
        hits = rerank(queries[q], cands, model=f"local:{MODEL}", style=style, max_docs_per_call=BATCH_DOCS if style == "batch" else K)
        torch.cuda.synchronize(); latencies.append(time.perf_counter() - t)
        ndcgs.append(ndcg10([shortlists[q][h.index] for h in hits], qrels[q]))
    results[style] = {"ndcg": sum(ndcgs) / len(ndcgs), "p50": pct(latencies, 50), "p95": pct(latencies, 95),
                      "wall": time.perf_counter() - t0}
    r = results[style]
    print(f"{style:6s}  nDCG@10 {bm25_ndcg:.3f} -> {r['ndcg']:.3f}   per query p50 {r['p50']:.2f} s, "
          f"p95 {r['p95']:.2f} s   wall {r['wall']:.0f} s")

In [ ]:
from IPython.display import Markdown
rows = [f"| {s} | {bm25_ndcg:.3f} | {r['ndcg']:.3f} | {r['ndcg'] - bm25_ndcg:+.3f} | {r['p50']:.2f} | {r['p95']:.2f} | {r['wall']:.0f} |"
        for s, r in results.items()]
Markdown(f"**{MODEL}** on {gpu}, {len(qids)} SciFact test queries, BM25 top-{K}, float16\n\n"
         "| shape | nDCG@10 BM25 | nDCG@10 reranked | Δ | p50 s/query | p95 s/query | wall s |\n|---|---|---|---|---|---|---|\n"
         + "\n".join(rows))

**About these latencies.** They are per query (all K candidates) on a free T4, with the checkpoint's reference
PyTorch code: the T4 has no bfloat16 and no FlashAttention, and the decider's optimized path (packed sequences,
CUDA graphs) needs an Ampere or newer GPU. They are **not comparable with hosted API latencies**, which run the same
kind of model on datacenter GPUs with batching across users, and include the network. For hosted numbers see
the [benchmark](https://hevmind.com/research/rerankers). Use this notebook to compare the *shapes* with one
another on one machine, and to see the quality.

**Why batch is split into requests of five here.** This checkpoint's reference code answers each extra question
in a request by re-running its linear-attention layers over the whole shared state, so on this path a batch
request's cost grows with the square of its length, and twenty documents in one request run out of the T4's
memory. Five per request is about as fast as it gets on a T4. Batch scores are absolute probabilities, so scores
from separate requests stay comparable. (Hosted, and on Ampere GPUs with the checkpoint's fast path, the state is
read once and batch costs about the same as choice.) Choice stays one request over all candidates: its
probabilities are only comparable within one list.

With fifteen queries the nDCG@10 differences between shapes are noise-sized; the
[rerank-bench](https://github.com/hev/rerank-bench) numbers use all 300 SciFact queries plus NFCorpus and FiQA.

## Thresholds: only pair and batch

Pair and batch scores are calibrated probabilities, so a threshold prunes candidates the model is confident are
irrelevant. Choice scores sum to one across the list, so `hev-rerank` refuses a threshold on them.

In [ ]:
q = qids[0]
cands = [corpus[d] for d in shortlists[q]]
kept = rerank(queries[q], cands, model=f"local:{MODEL}", style="batch", max_docs_per_call=BATCH_DOCS, threshold=0.1)
print(f"{queries[q]!r}: batch keeps {len(kept)} of {len(cands)} candidates at threshold 0.1")
try:
    rerank(queries[q], cands, model=f"local:{MODEL}", style="choice", threshold=0.1)
except ValueError as e:
    print("ValueError:", e)

## Optional: bigger models (not free tier)

The cell below is **off by default and will not fit a free T4**. It shows how to run larger open-weight decision
models with the same call, on a paid Colab GPU:

- [`Cloudflare/clef-flash`](https://huggingface.co/Cloudflare/clef-flash) (9B, Apache-2.0) in bfloat16 needs about
  20 GB: an **L4** (24 GB) or A100. Its best measured shape is pair.
- 27B models ([`Cloudflare/clef`](https://huggingface.co/Cloudflare/clef), Apache-2.0) in **4-bit** need about
  18 GB with `bitsandbytes`: an L4 or A100.
- `perplexity-ai/pplx-decider-v1.3` (27B) runs in bfloat16 on an 80 GB A100/H100.

This cell was not executed for this notebook's saved outputs.

In [ ]:
RUN_BIGGER_MODELS = False   # set True on an L4 / A100 runtime (Colab Pro); not free tier

if RUN_BIGGER_MODELS:
    del model; local._LOADED.clear(); torch.cuda.empty_cache()

    # Clef-flash 9B, bfloat16 on an L4 or A100.
    big = local.load("Cloudflare/clef-flash")
    for h in rerank(query, docs, model="local:Cloudflare/clef-flash", style="pair", client=big):
        print(f"  {h.score:6.3f}  [{h.index}] {docs[h.index][:90]}")
    del big; torch.cuda.empty_cache()

    # A 27B decision model in 4-bit (pip install bitsandbytes).
    from transformers import BitsAndBytesConfig
    q4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    big = local.load("Cloudflare/clef", quantization_config=q4)
    for h in rerank(query, docs, model="local:Cloudflare/clef", style="pair", client=big):
        print(f"  {h.score:6.3f}  [{h.index}] {docs[h.index][:90]}")